In [7]:
import torch
import json

with open("data/instruction_tuning.json") as f:
    data = json.load(f)

data[:3]

[{'instruction': 'Sam is heavier than Emma. Emma is heavier than David. Who is the lightest of the three?',
  'input': '',
  'output': 'The order is: Sam > Emma > David. David is the lightest.'},
 {'instruction': 'Fill in the missing number in the sequence: 11, ?, 17, 20, 23',
  'input': '',
  'output': 'The pattern increases by 3 each time. The missing number is 14.'},
 {'instruction': 'A class begins at 6:00 AM and runs for 1 hour and 45 minutes. When does it finish?',
  'input': '',
  'output': '6:00 AM + 1 hour and 45 minutes = 7:45 AM. The class finishes at 7:45 AM.'}]

In [3]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
import transformers
transformers.__version__

'5.11.0'

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'google/gemma-3-1b-pt'
DEVICE = 'mps'
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=DEVICE, torch_dtype='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model

Loading weights: 100%|██████████| 340/340 [00:02<00:00, 168.45it/s]


Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

In [17]:
input_text = "What is the capital of France?"
tokenizer(input_text)

{'input_ids': [2, 3689, 563, 506, 5279, 529, 7001, 236881], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [20]:
tokenizer(input_text, return_tensors="pt")

{'input_ids': tensor([[     2,   3689,    563,    506,   5279,    529,   7001, 236881]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [18]:
tokenizer.encode(input_text)

[2, 3689, 563, 506, 5279, 529, 7001, 236881]

In [22]:
prompt = "What is the capital of France?"
model_inputs = tokenizer.encode(prompt)
model_inputs

[2, 3689, 563, 506, 5279, 529, 7001, 236881]

In [34]:
def sum(a, b, **prams):
    return a + b 

inputs = {"a": 2, "b": 3, "c": 10}
sum(**inputs)

5

In [39]:
prompt = "What is the capital of France?"
model_inputs = tokenizer.encode(prompt)
torch.tensor([model_inputs])

tensor([[     2,   3689,    563,    506,   5279,    529,   7001, 236881]])

In [40]:
prompt = "What is the capital of France?"
model_inputs = tokenizer.encode(prompt)

with torch.inference_mode():
    generation = model.generate(inputs=torch.tensor(model_inputs), max_new_tokens=50, do_sample=False)
    # generation = generation[0][input_len:]

decoded = tokenizer.decode(generation, skip_special_tokens=True)
print(decoded)

IndexError: tuple index out of range

In [14]:
prompt = "What is the capital of France?"
model_inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

input_len = model_inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**model_inputs, max_new_tokens=50, do_sample=False)
    generation = generation[0][input_len:]

decoded = tokenizer.decode(generation, skip_special_tokens=True)
print(decoded)

23336

1209 + 21343 = 23336

1209 + 21343 = 23336

1209 + 


In [15]:
1209 + 21343

22552